In [0]:
# Load bronze tables into spark dataframes
airlines_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.airlines')
airports_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.airports')
flights_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.flights')
cancellation_codes_df = spark.read.table('`flight-delay-analytics-catalog`.bronze.cancellation_codes')

In [0]:
from pyspark.sql import functions as F

**1. Standardize data types**

In [0]:
# check schema of airlines_df
airlines_df.printSchema()

# check sample data of airlines_df
display(airlines_df.limit(5))

In [0]:
# check schema of airports_df
airports_df.printSchema()

# check sample data of airports_df
display(airports_df.limit(5))

In [0]:
# check schema of cancellation_codes_df
cancellation_codes_df.printSchema()

# check sample data of cancellation_codes_df
display(cancellation_codes_df.limit(5))

In [0]:
# check schema of flights_df
flights_df.printSchema()

# check sample data of flights_df
display(flights_df.limit(5))

In [0]:
# create one FLIGHT_DATE column from year, month, day columns
flights_df = flights_df.withColumn(
    'FLIGHT_DATE',
    F.make_date(
        F.col('YEAR'),
        F.col('MONTH'),
        F.col('DAY')
    )
)

In [0]:
# list of time columns
time_columns = [
    'SCHEDULED_DEPARTURE',
    'DEPARTURE_TIME',
    'WHEELS_OFF',
    'WHEELS_ON',
    'SCHEDULED_ARRIVAL',
    'ARRIVAL_TIME'
]

In [0]:
# create new datetime columns from integer time columns
for coln in time_columns:
    flights_df = flights_df.withColumn(
        f"{coln}_DATETIME",
    (
        F.try_to_timestamp(
        F.concat_ws(
            " ",
            F.col('FLIGHT_DATE'),
            F.lpad(F.col(coln).cast('string'), 4, '0')
        ),
        F.lit('yyyy-MM-dd HHmm')
    )
     )
    )


display(flights_df)

In [0]:
# check DIVERTED and CANCELLED unique values if convertible to boolean

display(flights_df.select(
    'DIVERTED'
).distinct())

display(flights_df.select(
    'CANCELLED'
).distinct())

In [0]:

# convert to appropriate data types
flights_df = flights_df.withColumn(
        'FLIGHT_NUMBER',
        F.col('FLIGHT_NUMBER').cast('string')
    ).withColumn(
        'CANCELLED',
        F.col('CANCELLED').cast('boolean')
    ).withColumn(
        'DIVERTED',
        F.col('DIVERTED').cast('boolean')
    )

flights_df.printSchema()


In [0]:
display(flights_df.limit(5))